# 13.8: Streaming responses

Most HTTP APIs stream using **server-sent events (SSE)**: the server keeps the connection open and writes a sequence of `data: {...}` lines, each carrying a JSON chunk with the next text increment, followed by a final marker. A client using the OpenAI Python library against any compatible server, such as the vLLM server started above, looks like this:


In [ ]:
import time
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed-locally")

start = time.perf_counter()
first, n_chunks = None, 0
stream = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Explain the KV cache in two sentences."}],
    temperature=0.7,
    max_tokens=200,
    stream=True,
)
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        first = first or time.perf_counter()
        n_chunks += 1
        print(delta, end="", flush=True)
end = time.perf_counter()
print(f"\nTTFT {first - start:.3f} s; {n_chunks} chunks in {end - first:.2f} s after the first")
